# DE1 — Data profile: UCI Online Retail II

**Project:** DS 5020 Mini-Project "Customers Like You" (PCA/SVD recommender).

This notebook *only profiles* the raw file `data/raw/online_retail_II.xlsx`. Nothing is cleaned, filtered, or modified here;
the goal is to understand what we are working with so that the cleaning decisions in the next step are evidence-based
and can be written up in `decision_log.md`.

Output artefact: `reports/data_profile.csv` (one row per column).

In [1]:
# Environment guard: this notebook must run in the "giftlane" conda env (pandas 3.0.6).
# It fails fast with "Wrong Python environment. Run: conda activate giftlane" otherwise.
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.env_check import check_env

print("environment OK:", check_env())

environment OK: /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python3.11


## 0. Setup

Imports, paths, and a record of the exact interpreter / pandas version used, so the results are reproducible.
Paths are resolved relative to the repo root so the notebook runs both from `notebooks/` and from the project root.

In [2]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "online_retail_II.xlsx"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

print("python :", sys.executable)
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("raw    :", RAW, "| exists:", RAW.exists())

python : /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python
pandas : 3.0.6
numpy  : 2.4.6
raw    : /Users/amiyasekhar/giftlane-recommender/data/raw/online_retail_II.xlsx | exists: True


## 1. Load every sheet and describe each one separately

The workbook has one sheet per fiscal year. We load them with `sheet_name=None` (returns a `dict` of DataFrames) and
report row count, column names, dtypes and the `InvoiceDate` range **per sheet**. Checking the sheets separately
(before concatenating) is what lets us see whether the sheets have the same schema and whether their date ranges overlap.

In [3]:
sheets = pd.read_excel(RAW, sheet_name=None)

for name, sdf in sheets.items():
    print(f"=== Sheet: {name!r} ===")
    print(f"rows        : {len(sdf):,}")
    print(f"columns     : {list(sdf.columns)}")
    print("dtypes      :")
    print(sdf.dtypes.to_string())
    print(f"InvoiceDate : {sdf['InvoiceDate'].min()}  ->  {sdf['InvoiceDate'].max()}")
    print()

=== Sheet: 'Year 2009-2010' ===
rows        : 525,461
columns     : ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
dtypes      :
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
InvoiceDate : 2009-12-01 07:45:00  ->  2010-12-09 20:01:00

=== Sheet: 'Year 2010-2011' ===
rows        : 541,910
columns     : ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
dtypes      :
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
InvoiceDate : 2010-12-01 08:26:00  ->  2011-12-09 12:50:00



## 2. Overlap check between the two sheets

If the date ranges of the two sheets overlap, the same transactions may have been exported twice. We

1. compute the overlap window `[max(starts), min(ends)]`,
2. count how many rows of **each** sheet fall inside that window, and
3. count how many of those rows are **exact duplicates across sheets** (every column equal, `NaN == NaN`).

For the cross-sheet comparison each row is reduced to a 64-bit hash of all its columns
(`pd.util.hash_pandas_object`), which treats missing values consistently and avoids a heavy multi-column merge.

**Nothing is dropped here** — this only reports the numbers so the decision can be made (and logged) in the cleaning step.

In [4]:
names = list(sheets)
assert len(names) == 2, f"expected 2 sheets, found {len(names)}: {names}"
name_a, name_b = names
a, b = sheets[name_a], sheets[name_b]
cols = list(a.columns)
assert cols == list(b.columns), "sheets have different columns"

a_min, a_max = a["InvoiceDate"].min(), a["InvoiceDate"].max()
b_min, b_max = b["InvoiceDate"].min(), b["InvoiceDate"].max()
ov_start, ov_end = max(a_min, b_min), min(a_max, b_max)

print(f"{name_a}: {a_min} -> {a_max}")
print(f"{name_b}: {b_min} -> {b_max}")

if ov_start > ov_end:
    print("\nDate ranges do NOT overlap.")
else:
    print(f"\nDate ranges OVERLAP: {ov_start} -> {ov_end}")

    a_in = a[a["InvoiceDate"].between(ov_start, ov_end)]
    b_in = b[b["InvoiceDate"].between(ov_start, ov_end)]
    print(f"rows of {name_a!r} inside overlap window : {len(a_in):,}")
    print(f"rows of {name_b!r} inside overlap window : {len(b_in):,}")
    print(f"distinct invoices in window, {name_a!r}  : {a_in['Invoice'].nunique():,}")
    print(f"distinct invoices in window, {name_b!r}  : {b_in['Invoice'].nunique():,}")

    # Exact cross-sheet duplicates: concatenate first so both halves share one dtype per column, then hash each row.
    both = pd.concat([a_in.assign(_sheet=name_a), b_in.assign(_sheet=name_b)], ignore_index=True)
    row_hash = pd.util.hash_pandas_object(both[cols], index=False).to_numpy()
    is_a = (both["_sheet"] == name_a).to_numpy()
    common = np.intersect1d(row_hash[is_a], row_hash[~is_a])

    n_a_dup = int(np.isin(row_hash[is_a], common).sum())
    n_b_dup = int(np.isin(row_hash[~is_a], common).sum())
    print(f"\ndistinct row-contents present in BOTH sheets          : {len(common):,}")
    print(f"rows of {name_a!r} that also appear exactly in {name_b!r}: {n_a_dup:,}")
    print(f"rows of {name_b!r} that also appear exactly in {name_a!r}: {n_b_dup:,}")

    if len(common):
        print("\nSample of cross-sheet exact duplicates (sorted so matching rows sit together):")
        sample = both[np.isin(row_hash, common)].sort_values(cols + ["_sheet"]).head(10)
        display(sample)

Year 2009-2010: 2009-12-01 07:45:00 -> 2010-12-09 20:01:00
Year 2010-2011: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00

Date ranges OVERLAP: 2010-12-01 08:26:00 -> 2010-12-09 20:01:00
rows of 'Year 2009-2010' inside overlap window : 22,523
rows of 'Year 2010-2011' inside overlap window : 22,523
distinct invoices in window, 'Year 2009-2010'  : 1,088
distinct invoices in window, 'Year 2010-2011'  : 1,088

distinct row-contents present in BOTH sheets          : 22,202
rows of 'Year 2009-2010' that also appear exactly in 'Year 2010-2011': 22,523
rows of 'Year 2010-2011' that also appear exactly in 'Year 2009-2010': 22,523

Sample of cross-sheet exact duplicates (sorted so matching rows sit together):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,_sheet
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,Year 2009-2010
22529,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,Year 2010-2011
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,Year 2009-2010
22528,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,Year 2010-2011
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2009-2010
22524,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2010-2011
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2009-2010
22527,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2010-2011
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2009-2010
22526,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,Year 2010-2011


## 3. Concatenate the sheets

Stack both sheets into one DataFrame (index reset). The README records 525,461 + 541,910 rows, so the total is
expected to be **1,067,371**. We check that number explicitly — a mismatch would mean a sheet was read incorrectly.

In [5]:
df = pd.concat(sheets.values(), ignore_index=True)
EXPECTED = 1_067_371
print(f"total rows : {len(df):,}")
print(f"expected   : {EXPECTED:,}  ->  match: {len(df) == EXPECTED}")
print(f"columns    : {list(df.columns)}")
print(f"memory     : {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

total rows : 1,067,371
expected   : 1,067,371  ->  match: True
columns    : ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


memory     : 227.1 MB


## 4. Column profile table

One row per column with dtype, missing count, % missing and the number of distinct values. This is the single
table to look at when deciding which columns need imputation/dropping (e.g. `Customer ID`) and which are
high-cardinality identifiers vs. low-cardinality categoricals. Saved to `reports/data_profile.csv`.

In [6]:
missing = df.isna().sum()
profile = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "missing": missing.to_numpy(),
        "pct_missing": (missing / len(df) * 100).round(3).to_numpy(),
        "unique": [df[c].nunique(dropna=True) for c in df.columns],
    }
)
profile.insert(0, "n_rows", len(df))

display(profile)

out_path = REPORTS / "data_profile.csv"
profile.to_csv(out_path, index=False)
print("saved ->", out_path.relative_to(ROOT))

,n_rows,column,dtype,missing,pct_missing,unique
0,1067371,Invoice,object,0,0.000,53628
1,1067371,StockCode,object,0,0.000,5305
2,1067371,Description,object,4382,0.411,5698
3,1067371,Quantity,int64,0,0.000,1057
4,1067371,InvoiceDate,datetime64[us],0,0.000,47635
5,1067371,Price,float64,0,0.000,2807
6,1067371,Customer ID,float64,243007,22.767,5942
7,1067371,Country,str,0,0.000,43


saved -> reports/data_profile.csv


## 5. Quick checks

Targeted questions that drive the cleaning rules in the next step:

- **`Customer ID` dtype** — identifiers should be integers / strings, so a float dtype is a red flag that needs explaining.
- **`StockCode` format** — the "normal" code is 5 digits optionally followed by letters (e.g. `85123A`). Anything else
  (`POST`, `D`, `M`, `BANK CHARGES`, …) is usually a fee/adjustment, not a product, and would pollute a product-space
  recommender.
- **Cancellations and non-positive values** — invoices starting with `C` are cancellations; `Quantity <= 0` and
  `Price <= 0` are returns, adjustments, or data errors.
- **Missing `Customer ID`** — rows without a customer cannot be used in a customer × product matrix at all.

### 5a. `Customer ID` dtype

In [7]:
cid = df["Customer ID"]
non_null = cid.dropna()
print("dtype                      :", cid.dtype)
print(f"missing values             : {cid.isna().sum():,} ({cid.isna().mean():.2%})")
print(f"non-missing values         : {len(non_null):,}")
print("all non-missing are whole numbers:", bool((non_null == np.floor(non_null)).all()))
print("min / max                  :", non_null.min(), "/", non_null.max())
print()
print("Why float64: the column has missing cells, and a NumPy int64 array cannot hold NaN, so pandas upcasts the")
print("whole column to float64 on load. The underlying IDs are integers (see check above); they display as 17850.0.")

dtype                      : float64
missing values             : 243,007 (22.77%)
non-missing values         : 824,364
all non-missing are whole numbers: True
min / max                  : 12346.0 / 18287.0

Why float64: the column has missing cells, and a NumPy int64 array cannot hold NaN, so pandas upcasts the
whole column to float64 on load. The underlying IDs are integers (see check above); they display as 17850.0.


### 5b. `StockCode` — numeric vs text, and codes that do not follow the `5 digits [+ letters]` pattern

In [8]:
sc_raw = df["StockCode"]
print("StockCode dtype:", sc_raw.dtype)
print("\nPython types stored in the column (object columns can hold a mix):")
print(sc_raw.map(lambda v: type(v).__name__).value_counts().to_string())

sc = sc_raw.astype("str").str.strip()
is_numeric_text = sc.str.fullmatch(r"\d+")
print(f"\nvalues that are purely digits            : {int(is_numeric_text.sum()):,} rows")
print(f"values containing at least one non-digit : {int((~is_numeric_text).sum()):,} rows")

PATTERN = r"\d{5}[A-Za-z]*"
standard = sc.str.fullmatch(PATTERN)
odd_mask = ~standard
print(f"\nStockCodes matching ^{PATTERN}$ : {int(standard.sum()):,} rows, {sc[standard].nunique():,} distinct codes")
print(f"StockCodes NOT matching          : {int(odd_mask.sum()):,} rows, {sc[odd_mask].nunique():,} distinct codes")

StockCode dtype: object

Python types stored in the column (object columns can hold a mix):


StockCode
int    932385
str    134986



values that are purely digits            : 932,385 rows
values containing at least one non-digit : 134,986 rows



StockCodes matching ^\d{5}[A-Za-z]*$ : 1,061,278 rows, 5,242 distinct codes
StockCodes NOT matching          : 6,093 rows, 62 distinct codes


In [9]:
odd = pd.DataFrame({"StockCode": sc[odd_mask], "Description": df.loc[odd_mask, "Description"]})

top30 = odd["StockCode"].value_counts().head(30)
most_common_desc = (
    odd.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .agg(lambda s: s.value_counts().index[0])
)
n_desc = odd.groupby("StockCode")["Description"].nunique()

top30_table = pd.DataFrame(
    {
        "rows": top30,
        "most_common_description": most_common_desc.reindex(top30.index),
        "n_distinct_descriptions": n_desc.reindex(top30.index),
    }
)
top30_table.index.name = "StockCode"
print("30 most frequent non-standard StockCodes:")
display(top30_table)

30 most frequent non-standard StockCodes:


,rows,most_common_description,n_distinct_descriptions
StockCode,,,
POST,2122,POSTAGE,1
DOT,1446,DOTCOM POSTAGE,1
M,1421,Manual,1
C2,282,CARRIAGE,1
D,177,Discount,1
S,104,SAMPLES,1
BANK CHARGES,102,Bank Charges,2
ADJUST,67,Adjustment by john on 26/01/2010 16,3
AMAZONFEE,43,AMAZON FEE,1


### 5c. Cancellations and non-positive `Quantity` / `Price`

In [10]:
inv = df["Invoice"].astype("str")
n_cancel_rows = int(inv.str.startswith("C").sum())
n_cancel_invoices = inv[inv.str.startswith("C")].nunique()
print(f"rows whose Invoice starts with 'C'     : {n_cancel_rows:,}")
print(f"distinct invoices starting with 'C'    : {n_cancel_invoices:,}")
print(f"rows with Quantity <= 0                : {int((df['Quantity'] <= 0).sum()):,}")
print(f"   of which Quantity < 0               : {int((df['Quantity'] < 0).sum()):,}")
print(f"rows with Price <= 0                   : {int((df['Price'] <= 0).sum()):,}")
print(f"   of which Price == 0                 : {int((df['Price'] == 0).sum()):,}")
print(f"   of which Price < 0                  : {int((df['Price'] < 0).sum()):,}")
print(f"rows with Quantity <= 0 AND not a 'C' invoice : {int(((df['Quantity'] <= 0) & ~inv.str.startswith('C')).sum()):,}")

rows whose Invoice starts with 'C'     : 19,494
distinct invoices starting with 'C'    : 8,292
rows with Quantity <= 0                : 22,950
   of which Quantity < 0               : 22,950
rows with Price <= 0                   : 6,207
   of which Price == 0                 : 6,202
   of which Price < 0                  : 5
rows with Quantity <= 0 AND not a 'C' invoice : 3,457


### 5d. Missing `Customer ID`

In [11]:
n_missing_cid = int(df["Customer ID"].isna().sum())
print(f"rows with missing Customer ID : {n_missing_cid:,} ({n_missing_cid / len(df):.2%} of all rows)")
print("missing Customer ID by sheet  :")
for name, sdf in sheets.items():
    print(f"   {name}: {int(sdf['Customer ID'].isna().sum()):,} of {len(sdf):,}")

rows with missing Customer ID : 243,007 (22.77% of all rows)
missing Customer ID by sheet  :
   Year 2009-2010: 107,927 of 525,461
   Year 2010-2011: 135,080 of 541,910


## Findings

_(to be filled in by hand after reviewing the outputs above)_

- **Sheets / schema:** …
- **Date overlap between sheets:** …
- **Exact cross-sheet duplicates:** …
- **Total rows after concatenation:** …
- **`Customer ID`:** …
- **`StockCode` anomalies:** …
- **Cancellations / non-positive quantity & price:** …
- **Missing `Customer ID`:** …
- **Implications for cleaning (to log in `decision_log.md`):** …